# Fine-tune DistilBERT — Behavioral Analytics Engine (IT23212640)

This notebook trains the **content threat model (Stage B1)** on the Jigsaw Toxic Comment dataset. It's multi-label, covering toxic, severe_toxic, obscene, **threat**, insult and identity_hate.

**Before you start:** Runtime → Change runtime type → **T4 GPU**.

Total time is about 1–1.5 hours. At the end you download `distilbert_model.zip` and unzip it into `behavior_engine/models/` on your laptop.

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Upload the project
Upload **`behavior_engine_colab.zip`**. It's in the delivered package and contains `src/`, `scripts/` and the prepared `data/processed/` files.

If you'd rather use Google Drive, put the zip in Drive and use the commented lines instead.

In [ ]:
from google.colab import files
up = files.upload()          # choose behavior_engine_colab.zip
# from google.colab import drive; drive.mount('/content/drive')
# !cp /content/drive/MyDrive/behavior_engine_colab.zip .
!unzip -q -o behavior_engine_colab.zip -d /content/
%cd /content/behavior_engine
!ls data/processed

## 3. Install libraries

In [ ]:
!pip -q install "transformers>=4.44" accelerate onnx onnxruntime pyarrow scikit-learn

## 4. Train
Default settings:
- 2 epochs, learning rate 3e-5, batch 32, max length 128
- threat examples oversampled ×4
- weighted BCE loss
- per-label thresholds tuned on validation
- ONNX + int8 export

After training, the script also evaluates on the Jigsaw test set, Davidson tweets and the moderation sanity set.

In [ ]:
!python scripts/train_distilbert.py --epochs 2

## 5. Results

In [ ]:
import json
r = json.load(open('reports/content_model_distilbert.json'))
print('Mean ROC-AUC (Jigsaw test):', r['jigsaw_test_mean_roc_auc'])
print('Abusive vs clean F1:', r['jigsaw_test_abusive_binary'])
print('Davidson cross-domain:', r['davidson_cross_domain'])
print('Latency (ms):', r['latency_ms_per_message'])
for l, v in r['jigsaw_test_per_label'].items():
    print(f"{l:14s} P={v['precision']:.3f} R={v['recall']:.3f} F1={v['f1']:.3f} AUC={v['roc_auc']:.3f}")

## 6. Download the trained model
Unzip this into `behavior_engine/` on your laptop. It creates `models/distilbert/` and `reports/content_model_distilbert.json`. The engine then uses DistilBERT automatically.

In [ ]:
!rm -rf models/distilbert/checkpoints
!zip -q -r distilbert_model.zip models/distilbert reports/content_model_distilbert.json
files.download('distilbert_model.zip')